In [1]:
# ==================================================
# LOAD DIM_BUILDING
# ==================================================

dim_building_df = spark.read.parquet(
    minio_path(
        "gold/data_model/dim_building"
    )
)


# ==================================================
# CREATE BRIDGE_PROPERTY_BUILDING
# Grain: 1 row = 1 canonical Property-Building relation
# ==================================================

bridge_property_building = (
    dim_building_df

    .filter(
        F.col("property_id").isNotNull()
    )

    .select(
        "property_id",
        "building_id",

        F.col("resolved_bbl")
        .alias("bbl"),

        "bin"
    )

    .withColumn(
        "relationship_type",
        F.lit("CANONICAL")
    )

    .dropDuplicates(
        [
            "property_id",
            "building_id"
        ]
    )
)

NameError: name 'spark' is not defined

In [2]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ==================================================
# PROJECT CONFIG
# ==================================================

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)


# ==================================================
# IMPORT PROJECT HELPERS
# ==================================================

from minio_config import configure_minio, minio_path


# ==================================================
# CREATE / GET SPARK SESSION
# ==================================================

spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Gold Property Building Bridge")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)


# ==================================================
# CONFIGURE MINIO
# ==================================================

configure_minio(spark)


# ==================================================
# REDUCE LOG NOISE
# ==================================================

spark.sparkContext.setLogLevel("WARN")


# ==================================================
# TEST
# ==================================================

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("MinIO helper loaded successfully")
print("Test:", spark.range(1).count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/07 18:13:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/07 18:13:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/07 18:13:52 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/07 18:13:52 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/09/07 18:13:52 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/09/07 18:13:52 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


Spark version: 3.4.0
Master: local[2]
MinIO helper loaded successfully
Test: 1


In [3]:
# ==================================================
# LOAD DIM_BUILDING
# ==================================================

dim_building_df = spark.read.parquet(
    minio_path(
        "gold/data_model/dim_building"
    )
)

print(
    "dim_building rows:",
    dim_building_df.count()
)

26/09/07 18:14:07 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


dim_building rows: 197958


In [4]:
# ==================================================
# CREATE BRIDGE_PROPERTY_BUILDING
# Grain: 1 row = 1 canonical Property-Building relation
# ==================================================

bridge_property_building = (
    dim_building_df

    .filter(
        F.col("property_id").isNotNull()
    )

    .select(
        "property_id",
        "building_id",
        F.col("resolved_bbl").alias("bbl"),
        "bin"
    )

    .withColumn(
        "relationship_type",
        F.lit("CANONICAL")
    )

    .dropDuplicates(
        ["property_id", "building_id"]
    )
)

print(
    "Bridge rows:",
    bridge_property_building.count()
)

print(
    "Distinct buildings:",
    bridge_property_building
    .select("building_id")
    .distinct()
    .count()
)

print(
    "Distinct properties:",
    bridge_property_building
    .select("property_id")
    .distinct()
    .count()
)

Bridge rows: 197459


Distinct buildings: 197459
Distinct properties: 171582


In [5]:
# ==================================================
# SAVE BRIDGE_PROPERTY_BUILDING
# ==================================================

BRIDGE_PATH = minio_path(
    "gold/data_model/bridge_property_building"
)

(
    bridge_property_building
    .write
    .mode("overwrite")
    .parquet(BRIDGE_PATH)
)

print("bridge_property_building saved successfully")
print("Path:", BRIDGE_PATH)

bridge_property_building saved successfully
Path: s3a://nyc-building-risk/gold/data_model/bridge_property_building


In [6]:
# ==================================================
# LOAD COMPLETE GOLD DATA MODEL
# ==================================================

dim_property = spark.read.parquet(
    minio_path("gold/data_model/dim_property")
)

dim_building = spark.read.parquet(
    minio_path("gold/data_model/dim_building")
)

bridge = spark.read.parquet(
    minio_path("gold/data_model/bridge_property_building")
)

fact_311 = spark.read.parquet(
    minio_path("gold/data_model/fact_311_event")
)

fact_hpd = spark.read.parquet(
    minio_path("gold/data_model/fact_hpd_violation")
)

fact_dob = spark.read.parquet(
    minio_path("gold/data_model/fact_dob_violation")
)

print("All Gold datasets loaded successfully")

All Gold datasets loaded successfully


In [7]:
# ==================================================
# ROW COUNTS
# ==================================================

print("dim_property:", dim_property.count())
print("dim_building:", dim_building.count())
print("bridge:", bridge.count())
print("fact_311:", fact_311.count())
print("fact_hpd:", fact_hpd.count())
print("fact_dob:", fact_dob.count())

dim_property: 858284
dim_building: 197958
bridge: 197459
fact_311: 885306
fact_hpd: 927308
fact_dob: 148688


In [8]:
# ==================================================
# PRIMARY KEY VALIDATION
# ==================================================

print(
    "dim_property distinct property_id:",
    dim_property.select("property_id").distinct().count()
)

print(
    "dim_building distinct building_id:",
    dim_building.select("building_id").distinct().count()
)

print(
    "bridge distinct pairs:",
    bridge.select(
        "property_id",
        "building_id"
    ).distinct().count()
)

print(
    "311 distinct event_id:",
    fact_311.select("event_id").distinct().count()
)

print(
    "HPD distinct hpd_event_id:",
    fact_hpd.select("hpd_event_id").distinct().count()
)

print(
    "DOB distinct dob_event_id:",
    fact_dob.select("dob_event_id").distinct().count()
)

dim_property distinct property_id: 858284
dim_building distinct building_id: 197958
bridge distinct pairs: 197459


311 distinct event_id: 885306
HPD distinct hpd_event_id: 927308
DOB distinct dob_event_id: 148688


In [9]:
# ==================================================
# ORPHAN KEY VALIDATION
# ==================================================

building_keys = (
    dim_building
    .select("building_id")
    .distinct()
)

property_keys = (
    dim_property
    .select("property_id")
    .distinct()
)


def count_building_orphans(df):
    return (
        df
        .filter(F.col("building_id").isNotNull())
        .select("building_id")
        .join(
            building_keys,
            on="building_id",
            how="left_anti"
        )
        .count()
    )


def count_property_orphans(df):
    return (
        df
        .filter(F.col("property_id").isNotNull())
        .select("property_id")
        .join(
            property_keys,
            on="property_id",
            how="left_anti"
        )
        .count()
    )


print("311 building orphans:", count_building_orphans(fact_311))
print("311 property orphans:", count_property_orphans(fact_311))

print("HPD building orphans:", count_building_orphans(fact_hpd))
print("HPD property orphans:", count_property_orphans(fact_hpd))

print("DOB building orphans:", count_building_orphans(fact_dob))
print("DOB property orphans:", count_property_orphans(fact_dob))

print("Bridge building orphans:", count_building_orphans(bridge))
print("Bridge property orphans:", count_property_orphans(bridge))

311 building orphans: 0
311 property orphans: 0
HPD building orphans: 0
HPD property orphans: 0
DOB building orphans: 0
DOB property orphans: 0
Bridge building orphans: 0
Bridge property orphans: 0


In [10]:
print(
    "Buildings without property_id:",
    dim_building
    .filter(F.col("property_id").isNull())
    .count()
)

print(
    "Buildings represented in bridge:",
    bridge
    .select("building_id")
    .distinct()
    .count()
)

Buildings without property_id: 499
Buildings represented in bridge: 197459
